# Building AI Agents with Persistent Memory using Mem0, Azure AI Agent Framework, and Azure AI Search

This notebook demonstrates how to build an intelligent travel booking agent that remembers user preferences across conversations. By combining Mem0, Azure AI Agent Framework, and Azure AI Search, we create an agent that provides personalized travel recommendations based on historical interactions.

## What You'll Learn:
1. **Mem0 Integration**: How to use Mem0 as a memory layer for AI agents
2. **Azure AI Search as Vector Store**: Store and retrieve memories using semantic search
3. **Persistent User Preferences**: Remember user preferences across different chat sessions
4. **Azure AI Agent Framework**: Build agents using Python Agent Framework with tool functions

## Prerequisites:
- Azure OpenAI deployment configured
- Azure AI Search service created
- Understanding of basic Azure AI Agent Framework concepts

## Understanding the Memory Architecture

### What is Mem0?

**Mem0** is an intelligent memory layer that provides:
- **Long-term Memory**: Store user preferences, past interactions, and learned information
- **Semantic Search**: Retrieve relevant memories based on context
- **User-specific Storage**: Maintain separate memory spaces for different users
- **Automatic Relevance**: Surface the most relevant memories for current context

### How the Components Work Together:
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  Azure AI       │────▶│      Mem0        │────▶│  Azure AI       │
│  Agent          │     │  Memory Layer    │     │  Search         │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                       │                         │
         │                       │                         │
    Processes              Stores/Retrieves          Vector Store
    User Input             User Preferences         for Memories &
                          & Context                  Travel Data
```

In [1]:
print("Hello, World!")

Hello, World!


In [ ]:
! pip install mem0ai azure-ai-projects

  Using cached h2-4.3.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached hyperframe-6.1.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached hpack-4.1.0-py3-none-any.whl.metadata (4.6 kB)
Using cached h2-4.3.0-py3-none-any.whl (61 kB)
Using cached hpack-4.1.0-py3-none-any.whl (34 kB)
Using cached hyperframe-6.1.0-py3-none-any.whl (13 kB)

   ---------------------------------------- 0/5 [hyperframe]
   -------- ------------------------------- 1/5 [hpack]
   -------- ------------------------------- 1/5 [hpack]
   -------- ------------------------------- 1/5 [hpack]
   ---------------- ----------------------- 2/5 [h2]
   ---------------- ----------------------- 2/5 [h2]
   ---------------- ----------------------- 2/5 [h2]
   ---------------- ----------------------- 2/5 [h2]
   ---------------- ----------------------- 2/5 [h2]
   ------------------------ --------------- 3/5 [qdrant-client]
   ------------------------ --------------- 3/5 [qdrant-client]
   ------------------------ ---------

## Import Required Packages

In [2]:
import json
import os
from typing import Annotated, List, Dict, Any
from datetime import datetime
import uuid

from IPython.display import display, HTML, Markdown
from dotenv import load_dotenv

# Azure AI Search
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchFieldDataType,
    SearchableField,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SearchField,
    VectorSearchAlgorithmMetric
)

# Mem0
from mem0 import Memory

# Azure AI Agent Framework
from azure.ai.projects import AIProjectClient
from azure.ai.agents.models import (
    Agent,
    AgentThread,
    ToolSet,
    FunctionTool,
    MessageTextContent,
    ThreadMessage,
    MessageRole,
    RunStatus,
    ThreadRun
)
from azure.identity import DefaultAzureCredential

## Environment Configuration

In [4]:
# Load environment variables
load_dotenv()

# Azure OpenAI Configuration
azure_openai_deployment = os.getenv("AZURE_AI_FOUNDRY_MODEL")
azure_openai_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")  # Use a recent API version


# Azure AI Search Configuration
search_service_endpoint = os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT")
search_api_key = os.getenv("AZURE_SEARCH_API_KEY")

# Index names
travel_index_name = "travel-hotels"
memory_index_name = "mem0-memories"

# Print all environment variables referenced in this cell
print("Environment Variables:")
print("=" * 50)
print(f"AZURE_AI_FOUNDRY_MODEL: {azure_openai_deployment}")
print(f"AZURE_AI_FOUNDRY_ENDPOINT: {azure_openai_endpoint}")
print(f"AZURE_AI_FOUNDRY_API_KEY: {'*' * 20 if azure_openai_api_key else None}")
print(f"AZURE_OPENAI_API_VERSION: {api_version}")
print(f"AZURE_SEARCH_SERVICE_ENDPOINT: {search_service_endpoint}")
print(f"AZURE_SEARCH_API_KEY: {'*' * 20 if search_api_key else None}")
print("=" * 50)


Environment Variables:
AZURE_AI_FOUNDRY_MODEL: gpt-4o
AZURE_AI_FOUNDRY_ENDPOINT: https://ibecfoundry.openai.azure.com/
AZURE_AI_FOUNDRY_API_KEY: ********************
AZURE_OPENAI_API_VERSION: 2024-02-01
AZURE_SEARCH_SERVICE_ENDPOINT: https://ibecsearch.search.windows.net
AZURE_SEARCH_API_KEY: ********************


## Initialize Azure AI Search for Travel Data

First, we'll set up Azure AI Search with sample hotel and destination data that our agent can search through.

In [5]:
# Initialize search clients
index_client = SearchIndexClient(
    endpoint=search_service_endpoint,
    credential=AzureKeyCredential(search_api_key)
)

# Create travel data index if it doesn't exist
travel_fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="name", type=SearchFieldDataType.String),
    SearchableField(name="description", type=SearchFieldDataType.String),
    SearchableField(name="location", type=SearchFieldDataType.String),
    SearchableField(name="amenities", type=SearchFieldDataType.String),
    SimpleField(name="price_per_night", type=SearchFieldDataType.Double),
    SimpleField(name="rating", type=SearchFieldDataType.Double),
    SearchableField(name="tags", type=SearchFieldDataType.String, collection=True)
]

travel_index = SearchIndex(name=travel_index_name, fields=travel_fields)

try:
    index_client.get_index(travel_index_name)
    print(f"✅ Index '{travel_index_name}' already exists")
except:
    index_client.create_index(travel_index)
    print(f"✅ Created index '{travel_index_name}'")

# Initialize search client for travel data
travel_search_client = SearchClient(
    endpoint=search_service_endpoint,
    index_name=travel_index_name,
    credential=AzureKeyCredential(search_api_key)
)

✅ Index 'travel-hotels' already exists


In [6]:
# Add sample travel data
sample_hotels = [
    {
        "id": "1",
        "name": "Le Meurice Paris",
        "description": "Luxury palace hotel with Michelin-starred dining and views of the Tuileries Garden",
        "location": "Paris, France",
        "amenities": "Spa, Michelin Restaurant, Concierge, Room Service, Fitness Center",
        "price_per_night": 850,
        "rating": 4.8,
        "tags": ["luxury", "romantic", "historic", "fine-dining", "spa"]
    },
    {
        "id": "2",
        "name": "Four Seasons Maui",
        "description": "Beachfront resort with world-class spa and family-friendly activities",
        "location": "Maui, Hawaii",
        "amenities": "Beach Access, Kids Club, Multiple Pools, Spa, Golf Course",
        "price_per_night": 695,
        "rating": 4.7,
        "tags": ["beach", "family-friendly", "resort", "spa", "golf"]
    },
    {
        "id": "3",
        "name": "Aman Tokyo",
        "description": "Minimalist luxury hotel with panoramic city views and traditional onsen",
        "location": "Tokyo, Japan",
        "amenities": "Onsen, City Views, Fine Dining, Spa, Business Center",
        "price_per_night": 780,
        "rating": 4.9,
        "tags": ["luxury", "business", "spa", "city", "minimalist"]
    },
    {
        "id": "4",
        "name": "Hotel Sacher Vienna",
        "description": "Historic hotel home of the original Sachertorte with elegant rooms",
        "location": "Vienna, Austria",
        "amenities": "Historic Cafe, Concierge, Accessible Rooms, Pet-Friendly",
        "price_per_night": 420,
        "rating": 4.6,
        "tags": ["historic", "accessible", "pet-friendly", "cultural", "cafe"]
    },
    {
        "id": "5",
        "name": "Fairmont Whistler",
        "description": "Ski-in/ski-out resort with family suites and mountain views",
        "location": "Whistler, Canada",
        "amenities": "Ski Access, Family Suites, Heated Pool, Kids Programs",
        "price_per_night": 380,
        "rating": 4.5,
        "tags": ["ski", "family-friendly", "mountain", "resort", "accessible"]
    }
]

# Upload hotels to search index
travel_search_client.upload_documents(documents=sample_hotels)
print(f"✅ Uploaded {len(sample_hotels)} hotels to search index")

✅ Uploaded 5 hotels to search index


## Initialize Mem0 with Azure AI Search

Now we'll configure Mem0 to use Azure AI Search as its vector store for persistent memory.

In [7]:
mem0_config = {
    "llm": {
        "provider": "azure_openai",
        "config": {
            "model": azure_openai_deployment,
            "temperature": 0.2,
            "max_tokens": 1500,
            "azure_kwargs": {
                "azure_deployment": azure_openai_deployment,
                "api_version": api_version,
                "azure_endpoint": azure_openai_endpoint,
                "api_key": azure_openai_api_key,
            }
        }
    },
    "vector_store": {
        "provider": "azure_ai_search",
        "config": {
            "service_name": search_service_endpoint.split("//")[1].split(".")[0],
            "api_key": search_api_key,
            "collection_name": "mem0",
            "embedding_model_dims": 1536
        }
    },
    "embedder": {
        "provider": "azure_openai",
        "config": {
            "model": "text-embedding-ada-002",  # Your embedding deployment name
            "azure_kwargs": {
                "azure_deployment": "text-embedding-ada-002",  # Update if different
                "api_version": api_version,
                "azure_endpoint": azure_openai_endpoint,
                "api_key": azure_openai_api_key,
            }
        }
    }
}

# Initialize Mem0
memory = Memory.from_config(mem0_config)
# Test the memory system
print("🧪 Testing Mem0 setup...")
test_messages = [
    {"role": "user", "content": "I prefer luxury hotels with spa services."},
    {"role": "assistant", "content": "I'll remember you prefer luxury hotels with spa services for future recommendations."}
]
memory.add(test_messages, user_id="test_user",
           metadata={"category": "preferences"})
test_memories = memory.get_all(user_id="test_user")
print(f"✅ Mem0 test successful! Found {len(test_memories)} memories")

🧪 Testing Mem0 setup...
✅ Mem0 test successful! Found 1 memories


In [11]:
test_memories

{'results': [{'id': '4f282a34-dd67-4e1c-9b7e-d9edcbba4d1b',
   'memory': 'Prefers luxury hotels with spa services',
   'hash': 'b7af1e635295620d37e156b22d228fe8',
   'metadata': {'category': 'preferences'},
   'created_at': '2025-11-04T00:39:35.046741-08:00',
   'updated_at': None,
   'user_id': 'test_user'}]}

## Create the Travel Booking Plugin

This plugin provides functions for searching hotels and managing user preferences through Mem0.

In [12]:
# Define tool functions for the Agent Framework
# These functions will be registered as tools that the agent can call

def search_hotels(query: str, max_results: int = 3) -> str:
    """Search for hotels based on criteria like location, amenities, or tags
    
    Args:
        query: Search query for hotels (location, amenities, etc.)
        max_results: Maximum number of results to return
        
    Returns:
        JSON string with list of hotels matching the search criteria
    """
    results = travel_search_client.search(
        search_text=query,
        top=max_results,
        include_total_count=True
    )

    hotels = []
    for result in results:
        hotels.append({
            "name": result["name"],
            "location": result["location"],
            "description": result["description"],
            "price_per_night": result["price_per_night"],
            "rating": result["rating"],
            "amenities": result["amenities"],
            "tags": result["tags"]
        })

    return json.dumps(hotels, indent=2)


def store_user_preference(user_id: str, preference: str) -> str:
    """Store user travel preferences and important information in memory
    
    Args:
        user_id: User identifier
        preference: User preference or information to remember
        
    Returns:
        Confirmation of stored preference
    """
    print(f"DEBUG: Storing preference for {user_id}: {preference}")

    try:
        # Simply add the preference to memory
        memory.add(preference, user_id=user_id)
        return f"✅ Stored: {preference}"
    except Exception as e:
        return f"❌ Error storing preference: {str(e)}"


def get_user_preferences(user_id: str) -> str:
    """Get all stored preferences for a user
    
    Args:
        user_id: User identifier
        
    Returns:
        All user preferences and memories
    """
    print(f"DEBUG: Getting all preferences for {user_id}")

    try:
        # Get all memories for the user
        results = memory.get_all(user_id=user_id)

        # Handle the dict response with 'results' key
        if isinstance(results, dict) and 'results' in results:
            results = results.get('results', [])

        if not results:
            return f"No preferences found for user {user_id}"

        # Format results
        memories = []
        for result in results:
            if isinstance(result, dict):
                memory_text = result.get('memory', str(result))
                memories.append(memory_text)
            else:
                memories.append(str(result))

        return f"User preferences for {user_id}:\n- " + "\n- ".join(memories)

    except Exception as e:
        print(f"ERROR getting preferences: {str(e)}")
        return f"No preferences found for user {user_id}"


def search_memories(user_id: str, query: str) -> str:
    """Search user's memories for relevant information
    
    Args:
        user_id: User identifier
        query: What to search for (e.g., 'family vacation', 'dietary restrictions')
        
    Returns:
        Relevant memories
    """
    print(f"DEBUG: Searching memories for {user_id} with query: '{query}'")

    try:
        # Let Mem0 handle the search and ranking
        results = memory.search(query, user_id=user_id)

        # Handle the dict response with 'results' key
        if isinstance(results, dict) and 'results' in results:
            results = results.get('results', [])

        if not results:
            return f"No memories found for query: {query}"

        # Format results
        memories = []
        for result in results:
            if isinstance(result, dict):
                memory_text = result.get('memory', str(result))
                # Include relevance score if available
                score = result.get('score', None)
                if score:
                    memories.append(f"{memory_text} (relevance: {score:.2f})")
                else:
                    memories.append(memory_text)
            else:
                memories.append(str(result))

        return "Relevant memories:\n- " + "\n- ".join(memories)

    except Exception as e:
        print(f"ERROR: {str(e)}")
        return "No memories found."

## Initialize the Azure AI Agent

Create our travel booking agent using Azure AI Agent Framework with access to the tool functions.

In [13]:
from azure.identity import AzureCliCredential
from agent_framework.azure import AzureOpenAIChatClient


# Create function tools for the agent
functions = FunctionTool(functions=[search_hotels, store_user_preference, get_user_preferences, search_memories])
toolset = ToolSet()
toolset.add(functions)

api_key = os.getenv("AZURE_AI_FOUNDRY_API_KEY")
foundry_openai_endpoint = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
model_name = os.getenv("AZURE_AI_FOUNDRY_MODEL", "gpt-4o")

if not api_key:
    raise ValueError(
        "AZURE_AI_FOUNDRY_API_KEY environment variable is required. "
        "Please set it in your .env file or environment."
    )

agent_chat_client= AzureOpenAIChatClient(
    endpoint=foundry_openai_endpoint,  # Azure OpenAI endpoint
    api_key=api_key,
    deployment_name=model_name  # Your deployment name
)

 
# Create the agent with instructions
travel_agent = agent_chat_client.create_agent(
    name="TravelBookingAssistant",
    instructions="""
    You are a personalized travel booking assistant with memory.
    
    WORKFLOW:
    1. When a user asks for help, search their memories using search_memories() with a relevant query
    2. Use the memories to personalize your response
    3. Store any new preferences they mention using store_user_preference()
    4. When the user is booking a new trip, first retrieve the user's general travel preferences by creating queries for hotels, dietary restrictions, location, amenities and budget. THEN use search_hotels() to find suitable options.
    5. Do not recommend hotels that are over budget. 
    
    IMPORTANT: For ALL memory operations (search_memories and store_user_preference), 
    you MUST use user_id='sarah_johnson_123' exactly as written.

    Example queries:
    - User asks about booking a trip → search_memories(query="preferences")
    - User asks about booking a trip → search_memories(query="dietary restrictions")
    - User asks about booking a trip → search_memories(query="location")
    - User asks about booking a trip → search_memories(query="amenities")
    - User asks about booking a trip → search_memories(query="budget")

    Always acknowledge what you found in their memories when responding.""",
    toolset=toolset
)

print(f"✅ Created agent: {travel_agent.id}")

✅ Created agent: 209d26ab-420a-40b9-8161-9ac2f68cb75c


## Helper Functions for Clean Display

In [14]:
def display_message(role: str, content: str, color: str = "#2E8B57", emoji: str = ""):
    """Display a message with nice formatting"""
    html = f"""
    <div style='
        margin: 10px 0; 
        padding: 15px 20px; 
        border-left: 4px solid {color}; 
        background: rgba(128, 128, 128, 0.05); 
        border-radius: 8px;
    '>
        <strong style='color: {color}; font-size: 16px;'>{emoji} {role}:</strong><br>
        <div style='margin-top: 10px; white-space: pre-wrap; font-size: 14px; line-height: 1.6;'>{content}</div>
    </div>
    """
    display(HTML(html))

def display_memory_operation(operation: str, details: str, color: str = "#9370DB"):
    """Display memory operations for educational purposes"""
    html = f"""
    <div style='
        margin: 5px 20px;
        padding: 10px 15px;
        background: rgba(147, 112, 219, 0.1);
        border: 1px solid {color};
        border-radius: 6px;
        font-family: monospace;
        font-size: 13px;
    '>
        <strong style='color: {color};'>🧠 Memory {operation}:</strong>
        <div style='margin-top: 5px; color: #555;'>{details}</div>
    </div>
    """
    display(HTML(html))

def display_function_call(function_name: str, args: dict, result: str = None):
    """Display function calls for transparency"""
    html = f"""
    <details style='margin: 5px 20px; padding: 10px; background: rgba(0, 123, 255, 0.05); border: 1px solid #007BFF; border-radius: 6px;'>
        <summary style='cursor: pointer; font-weight: bold; color: #007BFF;'>⚙️ Function Call: {function_name}</summary>
        <div style='margin-top: 10px; font-family: monospace; font-size: 12px;'>
            <div><strong>Arguments:</strong> {json.dumps(args, indent=2)}</div>
    """
    if result:
        html += f"<div style='margin-top: 10px;'><strong>Result:</strong><pre style='background: #f8f8f8; padding: 8px; border-radius: 4px; overflow-x: auto;'>{result}</pre></div>"
    html += "</div></details>"
    display(HTML(html))

## Demonstrate Travel Booking with Memory

Let's run through realistic travel booking scenarios showing how the agent remembers and uses user preferences.

### Scenario 1: First-time User - Anniversary Trip Planning

In [22]:
# User ID for our demonstration
sarah_user_id = "sarah_johnson_123"

print("🎯 SCENARIO 1: Sarah's First Booking - Anniversary Trip\n")

# Create a new thread for Sarah's conversation
sarah_thread = travel_agent.get_new_thread()

# First conversation
user_message1 = """Hi! I'm Sarah and I'm planning a special trip for my 10th wedding anniversary. 
We love romantic destinations, fine dining, and spa experiences. My husband has mobility issues, 
so we need accessible accommodations. Our budget is around $700-800 per night."""

display_message("Sarah", user_message1, "#4fc3f7", "👤")

# Send message and get response using the chat-based API
#response = travel_agent.run(
#    thread=sarah_thread,
#    messages=[{"role": "user", "content": user_message1}]
#)
response = await travel_agent.run(user_message1, thread=sarah_thread)
# Extract the response content
# Extract the response content from the AgentRunResponse object
# The response has a 'messages' attribute containing ChatMessage objects
if hasattr(response, 'messages') and response.messages:
    # Get the last message (assistant's response)
    last_message = response.messages[-1]
    # ChatMessage has a 'text' attribute, not 'content'
    response_content = last_message.text if hasattr(last_message, 'text') else str(last_message)
else:
    response_content = str(response)

display_message("Travel Assistant", response_content, "#81c784", "🤖")

🎯 SCENARIO 1: Sarah's First Booking - Anniversary Trip



In [25]:
user_message2 = """The Hotel Sacher sounds perfect! We're both vegetarian and I have a severe nut allergy. 
Can you tell me more about their dining options?"""

display_message("Sarah", user_message2, "#4fc3f7", "👤")

# Run the agent with the follow-up message
response2 = await travel_agent.run(user_message2, thread=sarah_thread)

# Extract the response content from the AgentRunResponse object
if hasattr(response2, 'messages') and response2.messages:
    # Get the last message (assistant's response)
    last_message2 = response2.messages[-1]
    # ChatMessage has a 'text' attribute
    response2_content = last_message2.text if hasattr(last_message2, 'text') else str(last_message2)
else:
    response2_content = str(response2)

display_message("Travel Assistant", response2_content, "#81c784", "🤖")

In [26]:

# After running all scenarios, verify memories are stored
from azure.search.documents import SearchClient
print("\n\n🔍 VERIFYING MEM0 STORAGE\n")

# Check Azure AI Search directly
mem0_search_client = SearchClient(
    endpoint=search_service_endpoint,
    index_name="mem0",
    credential=AzureKeyCredential(search_api_key)
)

try:
    # Count documents in the index
    results = mem0_search_client.search(
        search_text="*", include_total_count=True)
    total_docs = results.get_count()
    print(f"📊 Total documents in Mem0 index: {total_docs}")

    # Show first few documents
    print("\nSample documents:")
    for i, doc in enumerate(results):
        if i < 3:  # Show first 3
            print(f"\nDocument {i+1}:")
            print(f"  ID: {doc.get('id', 'N/A')}")
            print(f"  User ID: {doc.get('user_id', 'N/A')}")
            print(f"  Memory: {doc.get('payload', 'N/A')}")
except Exception as e:
    print(f"❌ Error checking Mem0 index: {str(e)}")
    print("The index might not be created yet or might be empty.")



🔍 VERIFYING MEM0 STORAGE

📊 Total documents in Mem0 index: 1

Sample documents:

Document 1:
  ID: 4f282a34-dd67-4e1c-9b7e-d9edcbba4d1b
  User ID: test_user
  Memory: {"category": "preferences", "user_id": "test_user", "data": "Prefers luxury hotels with spa services", "hash": "b7af1e635295620d37e156b22d228fe8", "created_at": "2025-11-04T00:39:35.046741-08:00"}


In [27]:
# Enhanced verification to debug Mem0 responses
print("🔍 ENHANCED MEM0 VERIFICATION\n")

# Create a unique test user
test_user = f"debug_user_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
test_memory = "I am vegetarian with a peanut allergy and love beach destinations"

print(f"1. Adding memory for {test_user}...")
add_result = memory.add(test_memory, user_id=test_user)
print(f"   Raw add result: {add_result}")

# Check if the result has a 'results' key
if isinstance(add_result, dict) and 'results' in add_result:
    actual_results = add_result.get('results', [])
    print(
        f"   Actual memories added: {len(actual_results) if isinstance(actual_results, list) else 0}")
    if isinstance(actual_results, list):
        for mem in actual_results:
            print(
                f"   - ID: {mem.get('id', 'N/A')}, Memory: {mem.get('memory', 'N/A')}")

print("\n2. Testing get_all()...")
all_mems = memory.get_all(user_id=test_user)
print(f"   Raw response: {all_mems}")
print(f"   Response type: {type(all_mems)}")

# Check if it's a dict with 'results' key
if isinstance(all_mems, dict):
    print(f"   Dict keys: {list(all_mems.keys())}")
    if 'results' in all_mems:
        results_value = all_mems['results']
        print(f"   'results' value type: {type(results_value)}")
        print(f"   'results' value: {results_value}")

        # If results is actually a list, show the memories
        if isinstance(results_value, list):
            print(f"   Number of memories: {len(results_value)}")
            for i, mem in enumerate(results_value):
                print(f"   Memory {i}: {mem}")

print("\n3. Testing search()...")
search_results = memory.search("peanut allergy", user_id=test_user)
print(f"   Raw response: {search_results}")
print(f"   Response type: {type(search_results)}")

# Check if it's a dict with 'results' key
if isinstance(search_results, dict):
    print(f"   Dict keys: {list(search_results.keys())}")
    if 'results' in search_results:
        results_value = search_results['results']
        print(f"   'results' value type: {type(results_value)}")
        print(f"   'results' value: {results_value}")

print("\n4. Testing direct API access...")
# Try to access memories through Azure AI Search directly
try:
    mem0_search_client = SearchClient(
        endpoint=search_service_endpoint,
        index_name="mem0",
        credential=AzureKeyCredential(search_api_key)
    )

    # Wait a moment for indexing
    import time
    time.sleep(2)

    # Search for the test user's memories
    azure_results = mem0_search_client.search(
        search_text="*",
        filter=f"user_id eq '{test_user}'",
        include_total_count=True
    )

    print(f"   Documents found in Azure: {azure_results.get_count()}")
    for doc in azure_results:
        payload = doc.get('payload', {})
        if isinstance(payload, str):
            import json
            try:
                payload = json.loads(payload)
            except:
                pass
        print(f"   - Memory: {payload}")

except Exception as e:
    print(f"   Error: {e}")

print("\n5. Testing Mem0 version...")
# Check if we need to use a different method or property
if hasattr(memory, '__version__'):
    print(f"   Mem0 version: {memory.__version__}")
if hasattr(memory, 'version'):
    print(f"   Mem0 version: {memory.version}")

# Check available methods
print("\n6. Available memory methods:")
for attr in dir(memory):
    if not attr.startswith('_') and callable(getattr(memory, attr)):
        print(f"   - {attr}")

🔍 ENHANCED MEM0 VERIFICATION

1. Adding memory for debug_user_20251104_140525...
   Raw add result: {'results': [{'id': '9dedae8e-ec11-4c51-9e03-67e07b9be691', 'memory': 'Is vegetarian', 'event': 'ADD'}, {'id': 'e65d55de-e949-48eb-a8b1-523c15aac27b', 'memory': 'Has a peanut allergy', 'event': 'ADD'}, {'id': '1363c24d-dbb6-40ff-9e13-3b0e28725d33', 'memory': 'Loves beach destinations', 'event': 'ADD'}]}
   Actual memories added: 3
   - ID: 9dedae8e-ec11-4c51-9e03-67e07b9be691, Memory: Is vegetarian
   - ID: e65d55de-e949-48eb-a8b1-523c15aac27b, Memory: Has a peanut allergy
   - ID: 1363c24d-dbb6-40ff-9e13-3b0e28725d33, Memory: Loves beach destinations

2. Testing get_all()...
   Raw response: {'results': [{'id': '1363c24d-dbb6-40ff-9e13-3b0e28725d33', 'memory': 'Loves beach destinations', 'hash': '2b8647bbe643bab15851b36fb878b85f', 'metadata': None, 'created_at': '2025-11-04T05:05:46.505192-08:00', 'updated_at': None, 'user_id': 'debug_user_20251104_140525'}, {'id': '9dedae8e-ec11-4c51-9

### Scenario 2: Return Visit - Family Vacation (Weeks Later)

In [29]:
print("\n\n🎯 SCENARIO 2: Sarah Returns Weeks Later for Family Vacation\n")
print("📅 Simulating time passing... Sarah starts a new conversation\n")

# Create a new thread to simulate a new conversation session
sarah_thread_new = travel_agent.get_new_thread()

user_message3 = "Hi, my husband and I are planning another trip. We are looking for a good hotel!"

display_message("Sarah", user_message3, "#4fc3f7", "👤")

# Run the agent with the message using the chat-based API
response3 = await travel_agent.run(user_message3, thread=sarah_thread_new)

# Extract the response content from the AgentRunResponse object
if hasattr(response3, 'messages') and response3.messages:
    # Get the last message (assistant's response)
    last_message3 = response3.messages[-1]
    # ChatMessage has a 'text' attribute
    response3_content = last_message3.text if hasattr(last_message3, 'text') else str(last_message3)
else:
    response3_content = str(response3)

display_message("Travel Assistant", response3_content, "#81c784", "🤖")



🎯 SCENARIO 2: Sarah Returns Weeks Later for Family Vacation

📅 Simulating time passing... Sarah starts a new conversation



In [ ]:
# Follow-up question
user_message4 = "Great suggestions! For the Maui option, what activities would you recommend for the kids?"

display_message("Sarah", user_message4, "#4fc3f7", "👤")

# Add message to thread
travel_agent.create_message(
    thread_id=sarah_thread_new.id,
    role=MessageRole.USER,
    content=user_message4
)

# Run the agent
run = travel_agent.create_run(
    thread_id=sarah_thread_new.id,
    agent_id=travel_agent.id
)

# Wait for completion
while run.status in [RunStatus.QUEUED, RunStatus.IN_PROGRESS]:
    import time
    time.sleep(1)
    run = travel_agent.get_run(thread_id=sarah_thread_new.id, run_id=run.id)

# Get the response
messages = travel_agent.list_messages(thread_id=sarah_thread_new.id)
latest_message = messages.data[0]
response4_content = latest_message.content[0].text.value if latest_message.content else ""

display_message("Travel Assistant", response4_content, "#81c784", "🤖")

DEBUG: Searching memories for sarah_johnson_123 with query: 'kids activities'


In [ ]:
print("\n🧪 TESTING MEMORY RETRIEVAL\n")

# First, ensure Sarah has some memories
test_preference = "I love romantic destinations with spa services"
result = store_user_preference(sarah_user_id, test_preference)
print(f"Store result: {result}")

# Now test retrieval
preferences = get_user_preferences(sarah_user_id)
print(f"\nRetrieved preferences:\n{preferences}")

# Also test the memory object directly
direct_memories = memory.get_all(user_id=sarah_user_id)
if isinstance(direct_memories, dict) and 'results' in direct_memories:
    direct_memories = direct_memories.get('results', [])
    
print(f"\nDirect memory.get_all() returned {len(direct_memories) if isinstance(direct_memories, list) else 0} memories")
if isinstance(direct_memories, list):
    for i, mem in enumerate(direct_memories):
        print(f"Memory {i}: {mem}")


🧪 TESTING MEMORY RETRIEVAL

DEBUG: Storing preference for sarah_johnson_123: I love romantic destinations with spa services
Store result: ✅ Stored: I love romantic destinations with spa services
DEBUG: Getting all preferences for sarah_johnson_123

Retrieved preferences:
User preferences for sarah_johnson_123:
- Loves romantic destinations for trips
- Vegetarian diet for dining preferences
- Budget is around $700-800 per night for travel accommodations
- Loves fine dining for trips
- Loves spa experiences for trips
- Severe nut allergy for dining preferences
- Requires accessible accommodations due to husband's mobility issues

Direct memory.get_all() returned 1 memories
Memory 0: results


In [26]:

# Check the Mem0 index structure in Azure AI Search
print("\n🔍 CHECKING MEM0 INDEX STRUCTURE\n")

try:
    # Get the mem0 index
    mem0_index = index_client.get_index("mem0")
    print("Mem0 index fields:")
    for field in mem0_index.fields:
        print(f"  - {field.name}: {field.type}")

    # Query the index directly
    mem0_search_client = SearchClient(
        endpoint=search_service_endpoint,
        index_name="mem0",
        credential=AzureKeyCredential(search_api_key)
    )

    # Get all documents for Sarah
    results = mem0_search_client.search(
        search_text="*",
        filter=f"user_id eq '{sarah_user_id}'",
        include_total_count=True
    )

    print(f"\nDocuments for {sarah_user_id}: {results.get_count()}")
    for doc in results:
        print(f"\nDocument ID: {doc.get('id')}")
        for key, value in doc.items():
            if key != 'id':
                print(
                    f"  {key}: {value[:100] if isinstance(value, str) and len(value) > 100 else value}")

except Exception as e:
    print(f"Error checking index: {str(e)}")


🔍 CHECKING MEM0 INDEX STRUCTURE

Mem0 index fields:
  - id: Edm.String
  - user_id: Edm.String
  - run_id: Edm.String
  - agent_id: Edm.String
  - vector: Collection(Edm.Single)
  - payload: Edm.String

Documents for sarah_johnson_123: 7

Document ID: 3bc0cab2-9820-4ec4-807b-7546f1e23dcf
  user_id: sarah_johnson_123
  agent_id: None
  run_id: None
  payload: {"user_id": "sarah_johnson_123", "data": "Loves romantic destinations with spa services", "hash": "5
  @search.score: 1.0
  @search.reranker_score: None
  @search.highlights: None
  @search.captions: None

Document ID: bcbc9e01-dc30-4292-b5ea-da044a5b305c
  user_id: sarah_johnson_123
  agent_id: None
  run_id: None
  payload: {"user_id": "sarah_johnson_123", "data": "Vegetarian diet for dining preferences", "hash": "7c50e1d1
  @search.score: 1.0
  @search.reranker_score: None
  @search.highlights: None
  @search.captions: None

Document ID: e2e39457-b435-43eb-bdc6-f331f80b7589
  user_id: sarah_johnson_123
  agent_id: None
  run_id

## Demonstrate Semantic Memory Search

Mem0's power comes from semantic search - finding relevant memories based on meaning, not just keywords.

In [27]:
print("🔍 SEMANTIC MEMORY SEARCH DEMONSTRATION\n")

# Search Sarah's memories for dietary-related information
dietary_search = memory.search(
    "dietary food allergies restrictions", user_id=sarah_user_id)

# Handle the dict response with 'results' key
if isinstance(dietary_search, dict) and 'results' in dietary_search:
    dietary_results = dietary_search.get('results', [])
else:
    dietary_results = dietary_search if isinstance(
        dietary_search, list) else []

print("Search Query: 'dietary food allergies restrictions'")
print(f"Results for Sarah:")
print("=" * 50)
if dietary_results:
    for mem in dietary_results:
        if isinstance(mem, dict):
            print(f"- {mem.get('memory', 'Unknown')}")
            print(f"  Relevance Score: {mem.get('score', 'N/A')}")
        else:
            print(f"- {mem}")
else:
    print("- No memories found")



🔍 SEMANTIC MEMORY SEARCH DEMONSTRATION

Search Query: 'dietary food allergies restrictions'
Results for Sarah:
- Severe nut allergy for dining preferences
  Relevance Score: 0.8828017
- Vegetarian diet for dining preferences
  Relevance Score: 0.86491835
- Requires accessible accommodations due to husband's mobility issues
  Relevance Score: 0.80573106
- Loves fine dining for trips
  Relevance Score: 0.8044828
- Loves spa experiences for trips
  Relevance Score: 0.786993
- Loves romantic destinations with spa services
  Relevance Score: 0.7807018
- Budget is around $700-800 per night for travel accommodations
  Relevance Score: 0.763523


## Key Takeaways

### 1. Persistent User Memory
- **Cross-Session Persistence**: User preferences are maintained across different conversations
- **User Isolation**: Each user has their own memory space
- **Automatic Context**: Agent automatically retrieves relevant memories

### 2. Mem0 Benefits
- **Semantic Understanding**: Retrieves memories based on meaning, not exact matches
- **Scalability**: Uses Azure AI Search for enterprise-grade storage
- **Privacy**: User memories are isolated and secure

### 3. Enhanced User Experience
- **No Repetition**: Users don't need to repeat preferences
- **Personalization**: Recommendations improve over time
- **Context Awareness**: Agent understands user history

## Summary

Congratulations! You've successfully built an AI travel agent with persistent memory capabilities using:

- **Mem0**: For intelligent, persistent memory management
- **Azure AI Search**: As a scalable vector store for memories and travel data
- **Azure AI Agent Framework**: To orchestrate the agent and tool functions

## What You've Learned:
1. How to integrate Mem0 with Azure AI Search for persistent memory
2. Building tool functions for Azure AI Agent Framework
3. Creating agents that remember user preferences across sessions
4. Using semantic search to retrieve relevant memories

## Real-World Applications:
- **Customer Service**: Remember customer history and preferences
- **Personal Assistants**: Maintain context across days or weeks
- **Healthcare**: Track patient information and preferences
- **Education**: Remember student progress and learning styles
- **E-commerce**: Personalized shopping based on history

## Next Steps:
- Implement memory expiration for time-sensitive information
- Add memory importance scoring
- Build multi-agent systems with shared memory
- Integrate with CRM systems for enterprise use
- Add memory versioning and audit trails